## Master SDF to many SDF converter

In [1]:
# Environment configurations
# pull in local configuration
%run config.py
# display file to screen
!cat config.py

# config.py
# This file wants to be listed in .gitignore

GNINA_LOC = "/home/dwaine/octoberproject/gnina"
GNINA_PARAMETER = "--no_gpu"


In [2]:
# Project configuration. Change to match your project.
projectdirname = "/decoys01"
master_sdf_filename = "generated_decoys_activeER_filtered_FINAL.sdf"

In [3]:
# Code configurations. Do not change.
from pathlib import Path
notebookdir = str(Path.cwd())

ligandrootdir = notebookdir + "/ligand"
rawdir =  "/raw"
processeddir = "/processed"

rawliganddir = ligandrootdir + projectdirname + rawdir
processedliganddir = ligandrootdir + projectdirname + processeddir
master_sdf = ligandrootdir + projectdirname + "/" + master_sdf_filename

In [4]:
# optional block used to create decoy ligands.

import pandas as pd
from rdkit.Chem import PandasTools

decoydf = None

decoydf = PandasTools.LoadSDF(
    master_sdf,
    molColName="ROMol",
    smilesName="SMILES",
    includeFingerprints=False,
    removeHs=False,
    strictParsing=True
)

# Add numbered ID column
decoydf["ID"] = [f"decoy{i+1}" for i in range(len(decoydf))]

print(decoydf.head())
print(decoydf.columns)
print(decoydf.shape)     # (rows, columns)
print(decoydf.info(memory_usage='deep'))

Failed to patch pandas - PandasTools will have limited functionality
Failed to patch pandas - unable to change molecule rendering


    Energy      ID                                             SMILES  \
0  3.68239  decoy1  [H]c1c([H])c2c(c([H])c1Br)C([H])([H])c1sc(N([H...   
1   21.029  decoy2  [H]c1c([H])c(-c2nc(N([H])[H])c3c([H])c([H])c(C...   
2  49.2558  decoy3  [H]c1c([H])c([H])c(C([H])([H])[H])c(N2C(=O)C([...   
3  93.6408  decoy4  [H]c1c(OC([H])([H])[H])c([H])c2c(N([H])[H])c(C...   
4  58.1729  decoy5  [H]c1nc(C(=O)c2nc(C([H])([H])[H])c([H])s2)c2c(...   

                                              ROMol  
0  <rdkit.Chem.rdchem.Mol object at 0x7689922989e0>  
1  <rdkit.Chem.rdchem.Mol object at 0x768992331f50>  
2  <rdkit.Chem.rdchem.Mol object at 0x768992331fc0>  
3  <rdkit.Chem.rdchem.Mol object at 0x768992332110>  
4  <rdkit.Chem.rdchem.Mol object at 0x7689923321f0>  
Index(['Energy', 'ID', 'SMILES', 'ROMol'], dtype='str')
(434, 4)
<class 'pandas.DataFrame'>
Index: 434 entries, 0 to 433
Data columns (total 4 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Energy 

In [10]:
from openbabel import openbabel as ob
from rdkit import Chem

def writeonedecoytoFS(row, tempfilename):
    
    # Your RDKit mol
    rdkit_mol = row['ROMol']

    # Convert RDKit MolBlock → Open Babel OBMol
    molblock = Chem.MolToMolBlock(rdkit_mol)
    ob_mol = ob.OBMol()
    ob_conversion = ob.OBConversion()
    ob_conversion.SetInFormat("mdl")  # MOL block format
    ob_conversion.ReadString(ob_mol, molblock)

    # Write out GNINA-ready SDF
    ob_conversion.SetOutFormat("sdf")
    sdf_string = ob_conversion.WriteString(ob_mol)

    with open(tempfilename, "w") as f:
        f.write(sdf_string)

In [15]:
# Iterate through master .sdf file and write individual Gnina friendly .sdf decoys.

#for idx, row in decoydf.head(5).iterrows(): #This version for testing just the first 5 rows
for idx, row in decoydf.iterrows():
    tempfilename = processedliganddir + "/" + row['ID'] + ".sdf"
    print("Write decoy to filesystem: " + tempfilename)
    writeonedecoytoFS(row, tempfilename)

Write decoy to filesystem: /home/dwaine/octoberproject/gninaworkflow/dock06/ligand/decoys01/processed/decoy1.sdf
Write decoy to filesystem: /home/dwaine/octoberproject/gninaworkflow/dock06/ligand/decoys01/processed/decoy2.sdf
Write decoy to filesystem: /home/dwaine/octoberproject/gninaworkflow/dock06/ligand/decoys01/processed/decoy3.sdf
Write decoy to filesystem: /home/dwaine/octoberproject/gninaworkflow/dock06/ligand/decoys01/processed/decoy4.sdf
Write decoy to filesystem: /home/dwaine/octoberproject/gninaworkflow/dock06/ligand/decoys01/processed/decoy5.sdf
Write decoy to filesystem: /home/dwaine/octoberproject/gninaworkflow/dock06/ligand/decoys01/processed/decoy6.sdf
Write decoy to filesystem: /home/dwaine/octoberproject/gninaworkflow/dock06/ligand/decoys01/processed/decoy7.sdf
Write decoy to filesystem: /home/dwaine/octoberproject/gninaworkflow/dock06/ligand/decoys01/processed/decoy8.sdf
Write decoy to filesystem: /home/dwaine/octoberproject/gninaworkflow/dock06/ligand/decoys01/proc

In [16]:
# Verify the count of files written to filesystem.
from pathlib import Path

sdf_count = len(list(Path(processedliganddir).glob("*.sdf")))
print("Files in " + processedliganddir + ": " + str(sdf_count))

Files in /home/dwaine/octoberproject/gninaworkflow/dock06/ligand/decoys01/processed: 434
